### **Hybrid Simulation Using PySD and Parallel Programming**

In [1]:
import psutil
import pysd
import pandas as pd
from joblib import Parallel, delayed

#### Identify number of cores in current device

In [2]:
print("Number of Physical Cores: ", psutil.cpu_count(logical=False))

Number of Physical Cores:  4


#### Pre-Translate (Done once by the main process)

In [3]:
# This creates the .py file version of your model so workers don't have to

model_filename = "Glossi v2.mdl"
py_model_file = pysd.read_vensim(model_filename)

# Get the path of the generated python file

py_path = py_model_file.py_model_file

#### Define a parallelable function

In [4]:
import pandas as pd
def run_single_simulation(num_steps) -> pd.DataFrame:
    """
    Each CPU core will now execute this function, load its own 
    copy of the model, run it, and then clear it from memory.
    """
    # Load the model INSIDE the function so it stays on this core
    local_model = pysd.load(py_path)
    
    # Run the simulation with the specific seed
    result = local_model.run(return_columns=["Dryness Level"], final_time=num_steps-1)
    
    return result["Dryness Level"]

#### Parallelizing the simulation

In [19]:
print(f"Starting single run...")

simulation_results = run_single_simulation(112) #set number of steps here

print(f"Completed single run.")
print(simulation_results.head())

Starting single run...
Completed single run.
time
0    4.915
1    4.495
2    4.722
3    4.302
4    4.529
Name: Dryness Level, dtype: float64


#### Save results to a CSV file

In [6]:
simulation_results.to_csv("single_run_results.csv")

#### Parse real-world data for Theil's *U*

In [26]:
real_df = pd.read_csv("glossi-dataset-v1.csv")

# create a column named "avg_dryness" based on "min_dryness" & "max_dryness"
real_df["avg_dryness"] = real_df["min_dryness"] + real_df["max_dryness"] / 2

# isolate "avg_dryness" column
avg_dryness_df = real_df["avg_dryness"]
print("shape of avg_dryness_df: ", avg_dryness_df.shape)
print("shape of simulation_results: ", simulation_results.shape)

shape of avg_dryness_df:  (112,)
shape of simulation_results:  (112,)


In [29]:
# convert simulation results and avg_dryness_df to numpy arrays
# simulation_results = simulation_results.to_numpy()
avg_dryness_arr = avg_dryness_df.to_numpy()

#### Conduct Theil's U

In [38]:
# Use the function to compare the simulated dryness levels with the real-world average dryness
from validation import theils_stats
metrics = theils_stats(avg_dryness_arr, simulation_results)

--- Theil's Decomposition Report ---
Total MSE: 1.8958
Um (Bias Proportion):      0.0698 (Ideal: close to 0)
Us (Variance Proportion):  0.6814 (Ideal: close to 0)
Uc (Covariation Prop):     0.2489 (Ideal: close to 1)
Sum check (Should be 1.0): 1.0000


### 💡Key Takeaways

Based on the model validation, we get the Theil's Statistics as `Us > Uc > Um`, meaning that the simulation model hovers in the same baseline as the real-world data, but greatly skews on the actual magnitude. In another perspective, the actual values of the model do not reflect the real-world, but behaves and moves like it. For future improvement, revisit the coefficients and recalibrate the scale of the simulation model. 